In [2]:
import geopandas as gpd
from matplotlib import pyplot as plt
import pandas as pd
import networkx as nx
import h3
from networkx.algorithms.community import louvain_communities
import igraph as ig
import leidenalg
import networkx as nx
import igraph as ig
import leidenalg
import geopandas as gpd
import pandas as pd
import h3
from tqdm import tqdm

import numpy as np
import random
seed = 505
random.seed(seed)        # or any integer
np.random.seed(seed)

In [3]:
# Load data
input_dir = "../data/output_files/retailcen_foursquare_mrc_2_mri_10_mrn_10_mtf_6_merge_n/"
retail_centres = gpd.read_parquet(input_dir + "3_afterfill.parquet")


In [4]:
#normalize the data

from sklearn.preprocessing import normalize

# Normalize POI vectors to unit length (row-wise L2 norm)
poi_cols = ['Dining and Drinking' , 'Leisure & Travel', 'Community & Health','Commerce']
retail_centres[poi_cols] = retail_centres[poi_cols].fillna(0)
poi_matrix = normalize(retail_centres[poi_cols].values, norm='l2', axis=1)
retail_centres['poi_vec'] = poi_matrix.tolist()


In [6]:
import pandas as pd
import numpy as np
import networkx as nx
import h3
from tqdm import tqdm

# Define POI thresholds to test
poi_thresholds = [1, 2, 3, 5, 10, 15, 20]  # Minimum foursquare_count required

def find_connected_components(h3_cells):
    """Find connected components among H3 cells using H3 adjacency"""
    if len(h3_cells) <= 1:
        return [list(h3_cells)] if h3_cells else []
    
    # Build adjacency graph
    G = nx.Graph()
    h3_set = set(h3_cells)
    
    for h3_cell in h3_cells:
        G.add_node(h3_cell)
        # Get immediate neighbors (ring 1)
        neighbors = h3.grid_ring(h3_cell, 1)
        for neighbor in neighbors:
            if neighbor in h3_set:
                G.add_edge(h3_cell, neighbor)
    
    # Find connected components
    components = list(nx.connected_components(G))
    return [list(component) for component in components]

def assign_removed_cells_to_components(removed_cells, component_cells_dict):
    """Assign each removed cell to the nearest component based on H3 adjacency"""
    if not removed_cells or not component_cells_dict:
        return {}
    
    # Create sets for faster lookup
    component_cell_sets = {comp_id: set(cells) for comp_id, cells in component_cells_dict.items()}
    assignments = {}
    unassigned_cells = set(removed_cells)
    
    # Iteratively expand search rings until all cells are assigned
    max_rings = 10  # Prevent infinite loops
    
    for ring_distance in range(1, max_rings + 1):
        newly_assigned = []
        
        for removed_cell in list(unassigned_cells):
            # Get neighbors at this ring distance
            try:
                neighbors = h3.grid_ring(removed_cell, ring_distance)
            except:
                # If grid_ring fails (edge cases), try smaller rings
                neighbors = []
                for r in range(1, ring_distance + 1):
                    try:
                        neighbors.extend(h3.grid_ring(removed_cell, r))
                    except:
                        continue
            
            # Check which component(s) these neighbors belong to
            component_matches = {}
            for neighbor in neighbors:
                for comp_id, comp_cells in component_cell_sets.items():
                    if neighbor in comp_cells:
                        if comp_id not in component_matches:
                            component_matches[comp_id] = 0
                        component_matches[comp_id] += 1
            
            # Assign to component with most neighbor connections
            if component_matches:
                best_component = max(component_matches.keys(), key=lambda x: component_matches[x])
                assignments[removed_cell] = best_component
                newly_assigned.append(removed_cell)
        
        # Remove newly assigned cells from unassigned set
        for cell in newly_assigned:
            unassigned_cells.remove(cell)
        
        # If all cells assigned, break
        if not unassigned_cells:
            break
    
    # Assign any remaining unassigned cells to component 0
    for cell in unassigned_cells:
        assignments[cell] = 0
    
    return assignments

def process_network_with_threshold(network_data, poi_threshold):
    """Process a single network with given POI threshold"""
    # Filter cells that meet the threshold
    high_poi_cells = network_data[network_data['foursquare_count'] >= poi_threshold]
    removed_cells = network_data[network_data['foursquare_count'] < poi_threshold]
    
    if len(high_poi_cells) == 0:
        # All cells removed - assign all to component 0
        result = network_data.copy()
        result[f'component_threshold_{poi_threshold}'] = 0
        return result
    
    # Find connected components among high POI cells
    high_poi_h3_list = high_poi_cells['h3_11'].tolist()
    components = find_connected_components(high_poi_h3_list)
    
    # Create mapping from h3 to component ID
    h3_to_component = {}
    for comp_id, component_cells in enumerate(components):
        for cell in component_cells:
            h3_to_component[cell] = comp_id
    
    # Assign removed cells to nearest components
    if len(removed_cells) > 0 and len(components) > 0:
        component_cells_dict = {i: comp for i, comp in enumerate(components)}
        removed_h3_list = removed_cells['h3_11'].tolist()
        
        removed_assignments = assign_removed_cells_to_components(
            removed_h3_list, component_cells_dict
        )
        
        # Add removed cell assignments to main mapping
        h3_to_component.update(removed_assignments)
    
    # Apply assignments to the network data
    result = network_data.copy()
    result[f'component_threshold_{poi_threshold}'] = result['h3_11'].map(h3_to_component).fillna(0).astype(int)
    
    return result

# Main processing loop
all_results = []
network_numbers = retail_centres['network_number'].value_counts().index.tolist()

print(f"Processing {len(network_numbers)} networks with {len(poi_thresholds)} POI thresholds...")

for network_number in tqdm(network_numbers, desc="Processing networks"):
    network_data = retail_centres[retail_centres['network_number'] == network_number].copy()
    
    # Process each POI threshold
    for poi_threshold in poi_thresholds:
        # Get reassigned version
        network_result = process_network_with_threshold(network_data, poi_threshold)
        reassigned_col = f'component_threshold_{poi_threshold}'
        network_data[reassigned_col] = network_result[reassigned_col]
        
        # Create filtered version (removed cells = -1)
        high_poi_mask = network_data['foursquare_count'] >= poi_threshold
        high_poi_cells = network_data[high_poi_mask]
        
        if len(high_poi_cells) == 0:
            # All cells removed - mark all as -1
            network_data[f'component_threshold_{poi_threshold}_filtered'] = -1
        else:
            high_poi_h3_list = high_poi_cells['h3_11'].tolist()
            components = find_connected_components(high_poi_h3_list)
            
            # Create mapping from h3 to component ID (only for high POI cells)
            h3_to_component = {}
            for comp_id, component_cells in enumerate(components):
                for cell in component_cells:
                    h3_to_component[cell] = comp_id
            
            # Apply assignments (high POI cells get component ID, low POI cells get -1)
            network_data[f'component_threshold_{poi_threshold}_filtered'] = network_data['h3_11'].map(h3_to_component).fillna(-1).astype(int)
    
    all_results.append(network_data)

# Combine all results
retail_centres_threshold_clustered = pd.concat(all_results, ignore_index=True)

# Save results with removed cells reassigned
output_filename = input_dir + "4_threshold_based_components.parquet"
retail_centres_threshold_clustered.to_parquet(output_filename, index=False)

print(f"Results with reassigned cells saved to {output_filename}")

# Combine all results
retail_centres_threshold_clustered = pd.concat(all_results, ignore_index=True)

# Save results
output_filename = input_dir + "4_threshold_based_components.parquet"
retail_centres_threshold_clustered.to_parquet(output_filename, index=False)

print(f"Results saved to {output_filename}")

# Display summary statistics
print("\nSummary of component counts by threshold:")
for threshold in poi_thresholds:
    reassigned_col = f'component_threshold_{threshold}'
    filtered_col = f'component_threshold_{threshold}_filtered'
    
    max_components_reassigned = retail_centres_threshold_clustered[reassigned_col].max() + 1
    max_components_filtered = retail_centres_threshold_clustered[filtered_col].max() + 1
    
    print(f"Threshold {threshold}: Max {max_components_reassigned} components (reassigned), Max {max_components_filtered} components (filtered)")

# Show example of how many cells were filtered at each threshold
print("\nCells meeting threshold requirements:")
for threshold in poi_thresholds:
    meeting_threshold = len(retail_centres[retail_centres['foursquare_count'] >= threshold])
    total_cells = len(retail_centres)
    percentage = (meeting_threshold / total_cells) * 100
    print(f"Threshold {threshold}: {meeting_threshold}/{total_cells} cells ({percentage:.1f}%)")

Processing 12044 networks with 7 POI thresholds...


Processing networks: 100%|██████████| 12044/12044 [04:06<00:00, 48.95it/s]


Results with reassigned cells saved to ../data/output_files/retailcen_foursquare_mrc_2_mri_10_mrn_10_mtf_6_merge_n/4_threshold_based_components.parquet
Results saved to ../data/output_files/retailcen_foursquare_mrc_2_mri_10_mrn_10_mtf_6_merge_n/4_threshold_based_components.parquet

Summary of component counts by threshold:
Threshold 1: Max 34 components (reassigned), Max 34 components (filtered)
Threshold 2: Max 29 components (reassigned), Max 29 components (filtered)
Threshold 3: Max 106 components (reassigned), Max 106 components (filtered)
Threshold 5: Max 221 components (reassigned), Max 221 components (filtered)
Threshold 10: Max 250 components (reassigned), Max 250 components (filtered)
Threshold 15: Max 183 components (reassigned), Max 183 components (filtered)
Threshold 20: Max 123 components (reassigned), Max 123 components (filtered)

Cells meeting threshold requirements:
Threshold 1: 183030/290139 cells (63.1%)
Threshold 2: 149821/290139 cells (51.6%)
Threshold 3: 105921/290

In [19]:
retail_centres_clustered

,h3_11,network_number,foursquare_count,Dining and Drinking,Leisure & Travel,Community & Health,Commerce,top_category,dominant_type,retail_landuse,...,total_foursquare_count,trackNodes,geometry,poi_vec,subcluster_leiden_r0.1_a0,subcluster_leiden_r0.1_a0.5,subcluster_leiden_r0.1_a1,subcluster_louvain_r0.1_a0,subcluster_louvain_r0.1_a0.5,subcluster_louvain_r0.1_a1
0,8b195da49442fff,33946,2,0.0,0.0,0.0,2.0,[Commerce],Commerce,False,...,38053,5387,"POLYGON ((528310.386 180570.629, 528290.045 18...","[0.0, 0.0, 0.0]",6,7,632,NaN,NaN,NaN
1,8b194ad347a1fff,33946,4,1.0,0.0,0.0,3.0,[Commerce],Commerce,True,...,38053,5387,"POLYGON ((533582.222 182911.838, 533561.9 1828...","[1.0, 0.0, 0.0]",3,6,3,NaN,NaN,NaN
2,8b194ad3666efff,33946,4,2.0,0.0,0.0,2.0,"[Dining and Drinking, Commerce]",Commerce,False,...,38053,5387,"POLYGON ((531754.321 182096.407, 531733.993 18...","[1.0, 0.0, 0.0]",5,4,6,NaN,NaN,NaN
3,8b195da4d632fff,33946,17,7.0,2.0,0.0,8.0,[Commerce],mixed,True,...,38053,5387,"POLYGON ((529121.351 182255.838, 529101.014 18...","[0.9615239476408232, 0.27472112789737807, 0.0]",10,10,2,NaN,NaN,NaN
4,8b195da4d15bfff,33946,2,1.0,1.0,0.0,0.0,"[Dining and Drinking, Leisure & Travel]",Dining and Drinking,False,...,38053,5387,"POLYGON ((529974.836 182678.664, 529954.503 18...","[0.7071067811865475, 0.7071067811865475, 0.0]",8,5,9,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
290134,8b195a8f1c51fff,67015,11,1.0,0.0,0.0,10.0,[Commerce],Commerce,False,...,11,1,"POLYGON ((270216.101 212037.043, 270195.004 21...","[1.0, 0.0, 0.0]",0,0,0,0.0,0.0,0.0
290135,8b1909a43d4dfff,3607,10,4.0,1.0,2.0,3.0,[Dining and Drinking],mixed,False,...,10,1,"POLYGON ((271531.989 665251.389, 271511.32 665...","[0.8728715609439696, 0.2182178902359924, 0.436...",0,0,0,0.0,0.0,0.0
290136,8b1909a480e8fff,3624,11,2.0,2.0,0.0,7.0,[Commerce],Commerce,False,...,11,1,"POLYGON ((274388.839 660374.072, 274368.174 66...","[0.7071067811865475, 0.7071067811865475, 0.0]",0,0,0,0.0,0.0,0.0
290137,8b1951ab0956fff,48696,11,7.0,2.0,1.0,1.0,[Dining and Drinking],Dining and Drinking,False,...,11,1,"POLYGON ((351657.797 395572.803, 351637.108 39...","[0.9525793444156804, 0.2721655269759087, 0.136...",0,0,0,0.0,0.0,0.0


In [111]:
#load the data and make a filtered version with only the top 100 networks
retail_centres_clustered = gpd.read_parquet(input_dir + "4_subclustered_multi_resolution.parquet")

In [113]:
#rename the network_numbers by size
network_numbers = retail_centres_clustered['network_number'].value_counts().index.tolist()
network_numbers = {k: i for i, k in enumerate(network_numbers)}
retail_centres_clustered['network_number'] = retail_centres_clustered['network_number'].map(network_numbers)
# Filter to only the top 100 networks
top_networks = retail_centres_clustered['network_number'].value_counts().nlargest(100).index.tolist()
retail_centres_top = retail_centres_clustered[retail_centres_clustered['network_number'].isin(top_networks)].copy()
#save the filtered data
retail_centres_top.to_parquet(input_dir + "4_subclustered_multi_resolution_top100.parquet", index=False)